# Number of assigned transcripts II

- using manual boundary-transcript overlap instead of adata.X
- slower but can quantify duplicate transcript assignments
- not used for main figure, but to assess duplicate transcript assignments (e.g. in Baysor)
- implemented using https://gustaveroussy.github.io/sopa/api/spatial/#sopa.spatial.assign_transcript_to_cell, also see https://github.com/gustaveroussy/sopa/issues/132

In [1]:
import os
import gc
import numpy as np
import yaml
import pandas as pd
import geopandas as gpd

In [2]:
import argparse
import logging
from pathlib import Path

In [3]:
import spatialdata as sd
import spatialdata_plot
from spatialdata import read_zarr
import sopa

/home/ubuntu/miniforge3/envs/seg_postprocessing/lib/python3.12/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(


In [4]:
import warnings
warnings.filterwarnings("ignore", message=".*ignoring keyword argument 'read_only'.*")

In [5]:
# noqa: E402
import sys
sys.path.insert(1, "/dss/dsshome1/0C/ra98gaq/Git/branch-metrics/cellseg-benchmark")
from cellseg_benchmark import BASE_PATH

## Helper functions

In [6]:
def compute_assigned_transcripts(
    *,
    cohort,
    base_path,
    shapes_keys=None,
    samples=None,
    overwrite=False,
    delete_temp=True,
    logger=None,
):
    """Compute and cache per-sample transcript assignments and transcript overlaps for a cohort."""

    output_dir = base_path / "metrics" / cohort / "assigned_transcripts" / "quantification_duplicate_assignments" / "cache"
    output_dir.mkdir(parents=True, exist_ok=True)

    meta = yaml.safe_load((base_path / "misc" / "sample_metadata.yaml").open())
    excluded = set(
        yaml.safe_load((base_path / "misc" / "samples_excluded.yaml").open()) or []
    )

    cohort_samples = [
        s for s, m in meta.items()
        if m["cohort"] == cohort and s not in excluded
    ]

    if samples is not None:
        cohort_samples = [s for s in cohort_samples if s in samples]

    if not cohort_samples:
        raise ValueError(f"No samples found for cohort='{cohort}'.")

    outs = []

    for sample in cohort_samples:
        f_assigned = output_dir / f"{sample}_assigned.csv"
        f_sum = output_dir / f"{sample}_summary.csv"

        if not overwrite and f_assigned.exists() and f_sum.exists():
            if logger:
                logger.info(f"[{sample}] using cached results")
            df_assigned = pd.read_csv(f_assigned)
            df_sum = pd.read_csv(f_sum)
        else:
            if logger:
                logger.info(f"[{sample}] loading sdata...")

            sdata = sd.read_zarr(
                base_path / "samples" / sample / "sdata_z3.zarr"
            )

            df_assigned, df_sum = compute_assigned_transcripts_single_sample(
                sdata=sdata,
                sample=sample,
                shapes_keys=shapes_keys,
                assign_fn=assign_fn,
                overlap_fn=overlap_fn,
                logger=logger,
            )

            df_assigned.to_csv(f_assigned, index=False)
            df_sum.to_csv(f_sum, index=False)

            del sdata
            gc.collect()

        outs.append((df_assigned, df_sum))

    df_assigned_all = pd.concat([a for a, _ in outs], ignore_index=True)
    df_sum_all = pd.concat([s for _, s in outs], ignore_index=True)

    df_assigned_all.to_csv(output_dir.parent / f"assigned_per_gene.csv", index=False)
    df_sum_all.to_csv(output_dir.parent / f"assigned_summary.csv", index=False)

    if delete_temp:
        for sample in cohort_samples:
            for suffix in ("assigned", "summary"):
                fp = output_dir / f"{sample}_{suffix}.csv"
                if fp.exists():
                    fp.unlink()

    return df_assigned_all, df_sum_all

In [7]:
def compute_assigned_transcripts_single_sample(
    sdata,
    sample,
    shapes_keys=None,
    points_key=None,
    *,
    assign_fn,
    overlap_fn,
    logger=None,
    overlap_predicate="within",
):
    """
    Run transcript-to-cell assignment for multiple segmentation methods (one sample),
    then summarize assignment rates and overlap ambiguity.

    Args:
        sdata (SpatialData): SpatialData containing transcript points and boundary shapes.
        sample (str): Sample name (used for labeling and default points_key).
        shapes_keys (Sequence[str], optional): Keys in `sdata.shapes`. If None, uses all shapes keys.
        points_key (str, optional): Key in `sdata.points`. Defaults to f"{sample}_transcripts".
        assign_fn (callable): Function like `assign_transcripts(sdata, points_key, shapes_key, logger=...)`.
        overlap_fn (callable): Function like `overlap_report_2d3d(sdata, points_key, shapes_key, ...)`.
        logger (logging.Logger, optional): Logger for progress output.
        overlap_predicate (str): Predicate passed to overlap_fn.

    Returns:
        df_assigned (pd.DataFrame): Per-gene assigned/unassigned counts per segmentation method.
        df_summary (pd.DataFrame): Per-method summary (assigned %, overlap ambiguity, etc.).
    """
    if points_key is None:
        points_key = f"{sample}_transcripts"
    if shapes_keys is None:
        shapes_keys = list(sdata.shapes.keys())

    records = []
    summary_records = []

    for i, shapes_key in enumerate(shapes_keys, start=1):
        seg_method = shapes_key.replace("boundaries_", "")

        shapes = sdata.shapes[shapes_key]
        shapes_zcol = "ZIndex" if "ZIndex" in shapes.columns else "layer"
        is_3d = (shapes_zcol in shapes.columns) and (shapes[shapes_zcol].nunique() > 1)
        dimension = "3D" if is_3d else "2D"

        if logger is not None:
            logger.info(f"[{i}] Quantifying transcript assignment in seg_method: '{seg_method}' ({dimension})")

        # remove previous assignment column if present
        meta_cols = getattr(sdata.points[points_key], "_meta", sdata.points[points_key]).columns
        if "cell_index" in meta_cols:
            cleaned = sdata.points[points_key].drop(columns=["cell_index"], errors="ignore")
            del sdata.points[points_key]
            sdata.points[points_key] = cleaned

        # assign transcripts -> cells
        assign_fn(sdata=sdata, points_key=points_key, shapes_key=shapes_key, logger=logger)

        # assigned vs unassigned per gene
        df = sdata.points[points_key].compute()
        df["assigned"] = ~df["cell_index"].isna()
        #df = sdata.points[points_key].assign(assigned=~sdata.points[points_key]["cell_index"].isna())

        counts = (
            df.groupby(["gene", "assigned"], observed=True)
              .size()
              .unstack(fill_value=0)
              .rename(columns={True: "assigned_count", False: "unassigned_count"})
              .reset_index()
        )
        if "assigned_count" not in counts.columns:
            counts["assigned_count"] = 0
        if "unassigned_count" not in counts.columns:
            counts["unassigned_count"] = 0

        assigned = int(counts["assigned_count"].sum())
        unassigned = int(counts["unassigned_count"].sum())
        total = assigned + unassigned

        # overlap ambiguity (summary only)
        ov_summary, _, _ = overlap_fn(
            sdata,
            points_key=points_key,
            shapes_key=shapes_key,
            target_sample=10_000_000,
            min_frac=0.2,
            random_state=0,
            predicate=overlap_predicate,
            logger=None,
        )
        ov = ov_summary.iloc[0]

        summary_row = {
            "sample": sample,
            "seg_method": seg_method,
            "dimension": dimension,
            "assigned_count": assigned,
            "unassigned_count": unassigned,
            "total_count": total,
            "pct_assigned": round((assigned / total) * 100, 2) if total else 0.0,
            "pct_transcripts_overlap_ge2": round(float(ov["pct_ambiguous_ge2"]) * 100, 4),
            "max_overlap_candidates": int(ov["max_candidates"]),
            "overlap_sample_n": int(ov["n_sample"]),
        }
        summary_records.append(summary_row)

        if logger is not None:
            logger.info(
                f"  [{seg_method}] assigned {assigned:,}/{total:,} ({summary_row['pct_assigned']}%, all transcripts)  "
                f"overlap ≥2 cells: {summary_row['pct_transcripts_overlap_ge2']}% "
                f"(of randomly sampled n={summary_row['overlap_sample_n']:,} transcripts)"
            )

        counts.insert(0, "sample", sample)
        counts.insert(1, "seg_method", seg_method)
        records.append(counts)

    df_assigned = pd.concat(records, ignore_index=True) if records else pd.DataFrame()
    df_assigned.columns.name = None
    df_summary = pd.DataFrame(summary_records)
    return df_assigned, df_summary

In [8]:
def assign_fn(sdata, points_key, shapes_key, logger=None):
    """Assign transcripts to cells using sopa.spatial.assign_transcript_to_cell() (3D capable).

    Args:
        sdata: SpatialData with `points[points_key]` and `shapes[shapes_key]`. For 3D, points
            need `global_z`; shapes need `ZIndex` or `layer`.
        points_key: Key in `sdata.points` for the transcript table.
        shapes_key: Key in `sdata.shapes` for the cell boundaries.
        logger: Optional logger for progress messages.

    Returns:
        None. Updates `sdata.points[points_key]` in-place by adding/overwriting `cell_index`.
    """
    # micron -> global (sopa expects global coords)
    t = sd.transformations.get_transformation(sdata.shapes[shapes_key], "micron")
    sd.transformations.set_transformation(sdata.shapes[shapes_key], t, to_coordinate_system="global")
    t = sd.transformations.get_transformation(sdata.points[points_key], "micron")
    sd.transformations.set_transformation(sdata.points[points_key], t, to_coordinate_system="global")

    # init/overwrite assignment column
    points_ddf = sdata.points[points_key]
    del sdata.points[points_key]
    points_ddf = points_ddf.drop(columns=["cell_id", "Unnamed: 0"], errors="ignore")
    points_ddf = points_ddf.assign(cell_index=np.nan)
    sdata.points[points_key] = points_ddf
    shapes = sdata.shapes[shapes_key]
    del sdata.shapes[shapes_key]

    if any(k.lower() in shapes_key.lower() for k in ("vpt", "merlin", "proseg")):
        shapes = shapes.drop(columns="cell_id", errors="ignore")
        shapes.index.name = None
    if shapes.crs is not None:
        shapes = shapes.set_crs(None, allow_override=True) # crs not compatible with sopa
    sdata.shapes[shapes_key] = shapes
    
    zcol = "ZIndex" if "ZIndex" in shapes.columns else "layer"
    
    if zcol not in shapes.columns or shapes[zcol].nunique() <= 1:
    # pure 2D
        sopa.spatial.assign_transcript_to_cell(
            sdata,
            points_key=points_key,
            shapes_key=shapes_key,
        )
        return

    # 3D: process per z-plane
    if shapes[zcol].nunique() > 1:
        if logger is not None:
            logger.info(f"Found {shapes[zcol].unique()} z-planes, computing boundary-transcript overlap in each separately...")
        for z in shapes[zcol].unique():
    
            z_shapes_key = f"{shapes_key}__z{z}"
            z_points_key = f"{points_key}__z{z}"
    
            sdata.shapes[z_shapes_key] = shapes[shapes[zcol] == z]
            sdata.points[z_points_key] = points_ddf[
                points_ddf["global_z"] == z
            ]
                
            sopa.spatial.assign_transcript_to_cell(
                sdata,
                points_key=z_points_key,
                shapes_key=z_shapes_key,
            )
    
            # write back assignments
            z_df = sdata.points[z_points_key][["cell_index"]].compute()
            z_df = z_df[~z_df.index.duplicated(keep="last")]
            z_map = z_df["cell_index"]
            
            sdata.points[points_key] = points_ddf.map_partitions(
                _patch,
                z_map,
                meta=points_ddf._meta,
                align_dataframes=False,
            )

            points_ddf = sdata.points[points_key]
            
            # cleanup
            del sdata.shapes[z_shapes_key]
            del sdata.points[z_points_key]
    
    sdata.points[points_key] = points_ddf.persist()

def _patch(part: pd.DataFrame, z_map: pd.Series) -> pd.DataFrame:
    m = part.index.isin(z_map.index)
    if m.any():
        part = part.copy()
        part.loc[m, "cell_index"] = z_map.loc[part.index[m]].to_numpy()
    return part

In [9]:
def overlap_fn(
    sdata,
    points_key,
    shapes_key,
    out_prefix=None,
    points_zcol="global_z",
    shapes_zcol=None,
    target_sample=10_000_000,
    min_frac=0.2,
    random_state=0,
    predicate="within",
    logger=None
):
    """Assess how many cell boundaries each transcript overlaps (2D or per-z 3D).

    Samples points, performs a spatial join, and counts candidate shapes per point.
    In 3D mode, joins are restricted to matching z-planes.

    Args:
        sdata: SpatialData with `points[points_key]` (needs `x`,`y`; and z-column for 3D)
            and `shapes[shapes_key]` (polygons; and z-column for 3D).
        points_key: Key of the points table.
        shapes_key: Key of the shapes table.
        out_prefix: If set, writes `<prefix>_summary.csv` and `<prefix>_dist.csv`.
        points_zcol: Z column in points (default `"global_z"`).
        shapes_zcol: Z column in shapes (auto-detected if None: "ZIndex" then "layer").
        target_sample: Target number of sampled points.
        min_frac: Minimum sampling fraction.
        random_state: Random seed.
        predicate: Spatial join predicate (e.g. `"within"`).
        logger: Optional logger.

    Returns:
        summary: One-row DataFrame with overlap statistics.
        dist: Distribution of candidate counts per sampled point.
        n_cand: Series of candidate counts per sampled point.
    """
    pts = sdata.points[points_key].drop(columns=["cell_id"], errors="ignore")
    shp = (
        sdata.shapes[shapes_key]
        .drop(columns=["cell_id"], errors="ignore")
        .set_crs(None, allow_override=True)
        .reset_index(drop=True)
    )

    if shapes_zcol is None:
        shapes_zcol = "ZIndex" if "ZIndex" in shp.columns else ("layer" if "layer" in shp.columns else None)

    is_3d = (
        shapes_zcol in shp.columns
        and shp[shapes_zcol].nunique() > 1
        and points_zcol in pts.columns
    )

    n_total = int(pts.shape[0].compute())
    frac = min(1.0, max(target_sample / max(n_total, 1), min_frac))
    sample = pts.sample(frac=frac, random_state=random_state).compute().reset_index(drop=False)

    def _counts(points_pdf, shapes_gdf):
        gpts = gpd.GeoDataFrame(
            points_pdf,
            geometry=gpd.points_from_xy(points_pdf["x"], points_pdf["y"]),
            crs=None,
        )
        j = gpts.sjoin(shapes_gdf, how="left", predicate=predicate)
        return j.groupby(j.index)["index_right"].apply(lambda s: s.notna().sum())

    if not is_3d:
        n_cand = _counts(sample, shp)
    else:
        parts = []
        for z, z_points in sample.groupby(points_zcol, sort=False):
            z_shapes = shp[shp[shapes_zcol] == z]
            parts.append(pd.Series(0, index=z_points.index) if z_shapes.empty else _counts(z_points, z_shapes))
        n_cand = pd.concat(parts).sort_index()

    n = int(n_cand.shape[0])
    n0 = int((n_cand == 0).sum())
    n1 = int((n_cand == 1).sum())
    n2p = int((n_cand >= 2).sum())
    max_cand = int(n_cand.max()) if n else 0

    summary = pd.DataFrame([{
        "points_key": points_key,
        "shapes_key": shapes_key,
        "predicate": predicate,
        "is_3d": bool(is_3d),
        "points_zcol": points_zcol if is_3d else "",
        "shapes_zcol": shapes_zcol if is_3d else "",
        "n_total": n_total,
        "n_sample": n,
        "frac": frac,
        "unassigned": n0,
        "unique": n1,
        "ambiguous_ge2": n2p,
        "pct_unassigned": (n0 / n) if n else 0.0,
        "pct_unique": (n1 / n) if n else 0.0,
        "pct_ambiguous_ge2": (n2p / n) if n else 0.0,
        "pct_ge5": (int((n_cand >= 5).sum()) / n) if n else 0.0,
        "pct_ge10": (int((n_cand >= 10).sum()) / n) if n else 0.0,
        "max_candidates": max_cand,
    }])

    dist = (
        n_cand.value_counts()
        .sort_index()
        .rename_axis("n_candidates")
        .reset_index(name="n_points")
    )
    dist["fraction"] = dist["n_points"] / n if n else 0.0

    if logger is not None:
        logger.info(
            f"sample={n:,}/{n_total:,} (frac={frac:.6f})  "
            f"unassigned={n0:,} ({(n0/n if n else 0):.3%})  "
            f"unique={n1:,} ({(n1/n if n else 0):.3%})  "
            f"ambiguous(>=2)={n2p:,} ({(n2p/n if n else 0):.3%})  "
            f"max_candidates={max_cand}"
        )
    
        preview = dist.head(25).set_index("n_candidates")[["n_points", "fraction"]].to_string()
        logger.info("\n" + preview)
    
        if len(dist) > 25:
            logger.info("...")

    if out_prefix:
        summary.to_csv(f"{out_prefix}_summary.csv", index=False)
        dist.to_csv(f"{out_prefix}_dist.csv", index=False)

    return summary, dist, n_cand

## Load sdata

In [10]:
parser = argparse.ArgumentParser(description="metrics_assigned_transcripts")
parser.add_argument("sample", help="Sample, e.g., 'foxf2_s2_r1'")
args = parser.parse_args(["aging_s8_r0"])
print(args)

Namespace(sample='aging_s8_r0')


In [11]:
# Logger setup
logger = logging.getLogger("compute_assigned_transcripts")
logger.setLevel(logging.INFO)
handler = logging.StreamHandler()
handler.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s]: %(message)s"))

In [12]:
sdata = sd.read_zarr(Path(BASE_PATH) / "samples" / args.sample / "sdata_z3.zarr")

version mismatch: detected: RasterFormatV02, requested: FormatV04


## Run

In [14]:
shapes_keys= list(sdata.shapes.keys())[::10]
shapes_keys = [shapes_keys[1]]
shapes_keys

['boundaries_Cellpose_1_DAPI_Transcripts']

In [ ]:
%%time
df_assigned, df_summary = compute_assigned_transcripts(
    cohort="aging",
    base_path=Path(BASE_PATH),
    shapes_keys=shapes_keys,
    samples=["aging_s8_r1", "aging_s8_r2", "aging_s10_r1"],
    overwrite=True,
    delete_temp=False,
    logger=logger,
)

version mismatch: detected: RasterFormatV02, requested: FormatV04


In [31]:
df_assigned

,sample,seg_method,gene,unassigned_count,assigned_count
0,aging_s8_r0,Cellpose_1_DAPI_Transcripts,A830036E02Rik,1965,33498
1,aging_s8_r0,Cellpose_1_DAPI_Transcripts,AI593442,18603,211465
2,aging_s8_r0,Cellpose_1_DAPI_Transcripts,Abca1,5418,55678
3,aging_s8_r0,Cellpose_1_DAPI_Transcripts,Abcb1a,4890,49716
4,aging_s8_r0,Cellpose_1_DAPI_Transcripts,Abcb1b,808,5755
...,...,...,...,...,...
545,aging_s8_r0,Cellpose_1_DAPI_Transcripts,Wnt7a,2260,15184
546,aging_s8_r0,Cellpose_1_DAPI_Transcripts,Xirp1,550,3854
547,aging_s8_r0,Cellpose_1_DAPI_Transcripts,Zcchc14,9467,56838
548,aging_s8_r0,Cellpose_1_DAPI_Transcripts,Zeb1,24911,159006


In [32]:
df_summary

,sample,seg_method,dimension,assigned_count,unassigned_count,total_count,pct_assigned,pct_transcripts_overlap_ge2,max_overlap_candidates,overlap_sample_n
0,aging_s8_r0,Cellpose_1_DAPI_Transcripts,2D,40563319,6483303,47046622,86.22,0.0008,2,9999997
